In [88]:
import os
print(os.getcwd())

/Users/raresolteanu/Desktop/WC-2026


In [89]:
os.chdir("/Users/raresolteanu/Desktop/WC-2026")
print(os.getcwd())

/Users/raresolteanu/Desktop/WC-2026


In [90]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## 1. Hard coded thresholds

In [91]:
#If the bookmakers say a team has an 82% or higher chance of winning,
#the model is blocked from predicting an upset because the favorite is just too strong.
MAX_FAVORITE_THRESHOLD = 0.82  
HIGH_MODEL_CONFIDENCE = 0.60
#If our model calculates that an underdog has a 9% higher chance (or a draw has an 11% higher chance) of happening
#  than what the bookmaker odds suggest, it triggers a prediction override
#change this values if you want the model be more based on the trained games or the odds please
#these were my personal decisions
UPSET_EDGE_THRESHOLD = 0.09  # 9% value edge required to predict underdog win
DRAW_EDGE_THRESHOLD = 0.11  # 14% value edge required to predict a surprise draw

## 2. Define features

In [92]:
baseline_features = [
    "elo_diff", "fifa_rank_diff", "fifa_points_zscore_diff",
    "form_score_5_diff", "form_score_10_diff", "form_trend_diff",
    "attack_diff", "defense_diff", "rest_days_diff",
    "neutral", "is_home_country", "is_home_confed", "is_away_confed",
    "missing_fifa_rank", "missing_fifa_points_zscore", "squad_value_diff", 
    "top_player_value_diff"

]

h2h_streak_features = [
    "h2h_matches_played", "h2h_wins_diff", "h2h_draws", "h2h_goals_diff", "h2h_avg_goals_diff",
    "home_win_streak", "home_unbeaten_streak", "away_win_streak", "away_unbeaten_streak",
    "win_streak_diff", "unbeaten_streak_diff"
]

advanced_features = [
    "strength_rating_diff", 
    "xg_balance_diff", 
    "missing_player_value_diff"
]

full_features = baseline_features + h2h_streak_features + advanced_features
odds_features = [
    "market_home_prob_norm", "market_draw_prob_norm", "market_away_prob_norm",
    "odds_prob_diff", "market_favorite_prob", "market_draw_prob"
]

all_features = full_features + odds_features


## 3. Load & Preprocess Datasets

### data loading

In [93]:
full_df = pd.read_csv("datasets/wc_2026_training_table_clean_with_spi_and_odds.csv")
schedule_2026 = pd.read_csv("datasets/WC 2026 match schedule advanced.csv")
odds_df = pd.read_csv("datasets/wc_2026_odds_experiment_model_table.csv")

### The added market and player value

In [94]:
from sklearn.linear_model import HuberRegressor
import warnings
warnings.filterwarnings('ignore')

squad_val_df = pd.read_csv("datasets/wc_2026_squad_market_values.csv")
player_val_df = pd.read_csv("datasets/wc_2026_top_players_market_values.csv")
strength_df = pd.read_csv("datasets/Team strength ratings.csv")

merged_impute = pd.merge(squad_val_df, strength_df[['team', 'elo_rating']], on='team', how='inner')
elo_to_squad = HuberRegressor(max_iter=2000)
elo_to_squad.fit(merged_impute[['elo_rating']].values, merged_impute['squad_value_eur_m'])

merged_player_impute = pd.merge(player_val_df, strength_df[['team', 'elo_rating']], on='team', how='inner')
elo_to_player = HuberRegressor(max_iter=2000)
elo_to_player.fit(merged_player_impute[['elo_rating']].values, merged_player_impute['market_value_eur_m'])

squad_val_dict = squad_val_df.set_index("team")["squad_value_eur_m"].to_dict()
player_val_dict = player_val_df.set_index("team")["market_value_eur_m"].to_dict()
elo_map = strength_df.set_index("team")["elo_rating"].to_dict()
median_elo = strength_df["elo_rating"].median()

def get_squad_value(team_name):
    if team_name in squad_val_dict:
        return squad_val_dict[team_name]
    elo = elo_map.get(team_name, median_elo)
    predicted = elo_to_squad.predict([[elo]])[0]
    return max(0.1, float(predicted))

def get_player_value(team_name):
    if team_name in player_val_dict:
        return player_val_dict[team_name]
    elo = elo_map.get(team_name, median_elo)
    predicted = elo_to_player.predict([[elo]])[0]
    return max(0.01, float(predicted))

for df in [full_df, odds_df, schedule_2026]:
    df["home_squad_val"] = df["home_team"].apply(get_squad_value)
    df["away_squad_val"] = df["away_team"].apply(get_squad_value)
    df["squad_value_diff"] = df["home_squad_val"] - df["away_squad_val"]
    
    df["home_top_player_val"] = df["home_team"].apply(get_player_value)
    df["away_top_player_val"] = df["away_team"].apply(get_player_value)
    df["top_player_value_diff"] = df["home_top_player_val"] - df["away_top_player_val"]

### Team Name Spellcheck & Alignment Check

In [95]:
if "is_placeholder_match" in schedule_2026.columns:
    real_schedule = schedule_2026[schedule_2026["is_placeholder_match"] == False]
else:
    real_schedule = schedule_2026
    
# Secondary filter: ignore team names containing placeholder keywords
real_schedule = real_schedule[
    (~real_schedule["home_team"].str.contains("Winner|Runner-up|Loser|3rd|Group|Match", na=False)) &
    (~real_schedule["away_team"].str.contains("Winner|Runner-up|Loser|3rd|Group|Match", na=False))
]
train_teams = set(full_df["home_team"].unique()).union(set(full_df["away_team"].unique()))
sched_teams = set(real_schedule["home_team"].unique()).union(set(real_schedule["away_team"].unique()))
unmatched_teams = sched_teams - train_teams
if unmatched_teams:
    print(f"The following {len(unmatched_teams)} teams in the schedule have NO historical matches: {unmatched_teams}")
    print("  This spelling mismatch will cause the model to ignore their historical records. Please check your data.")
else:
    print("All teams in the schedule have matching historical records!")

All teams in the schedule have matching historical records!


### Cleaning Function


In [96]:
def advanced_clean_and_verify(df, features, is_schedule=False):
    df = df.copy()


    #Add missing columns and set default values
    for col in features:
        if col not in df.columns:
            df[col] = np.nan


    # Fill NaNs based on feature type
    for col in features:
        if col in ["neutral"]:
            df[col] = df[col].fillna(1.0 if is_schedule else 0.0) # World Cup matches default to neutral
        elif col in ["is_home_country", "is_home_confed", "is_away_confed"]:
            df[col] = df[col].fillna(0.0) # Default to no home country/confed advantage
        else:
            df[col] = df[col].fillna(0.0)


    #Infinity values removal and Type Casting
    df[features] = df[features].replace([np.inf, -np.inf], np.nan)
    df[features] = df[features].fillna(0.0)
    df[features] = df[features].astype(np.float64)


    #Outlier clipping to prevent distorted predictions
    if "elo_diff" in df.columns:
        df["elo_diff"] = df["elo_diff"].clip(-1000.0, 1000.0)
    if "fifa_rank_diff" in df.columns:
        df["fifa_rank_diff"] = df["fifa_rank_diff"].clip(-210.0, 210.0)
    if "rest_days_diff" in df.columns:
        df["rest_days_diff"] = df["rest_days_diff"].clip(-20.0, 20.0)


    # Market Probability normalization (Ensure they sum to exactly 1.0)
    prob_cols = ["market_home_prob_norm", "market_draw_prob_norm", "market_away_prob_norm"]
    if all(col in df.columns for col in prob_cols):
        # Clip probabilities to [0.01, 0.99] to prevent 0 or negative values
        for col in prob_cols:
            df[col] = df[col].clip(0.01, 0.99)
            
        prob_sum = df[prob_cols].sum(axis=1)
        for col in prob_cols:
            df[col] = df[col] / prob_sum
    return df

In [97]:
full_model = advanced_clean_and_verify(full_df, full_features, is_schedule=False)
odds_model = advanced_clean_and_verify(odds_df, odds_features, is_schedule=False)
schedule_model = advanced_clean_and_verify(schedule_2026, full_features + odds_features, is_schedule=True)

full_all = full_model[(full_model["date"] < "2026-06-11") & (full_model["result"].notna())].copy()
print("Dataframes prepared successfully and cleaned!")

Dataframes prepared successfully and cleaned!


## 4. Train Upgraded Random Forest Model

# Super important: Why we choose as the final model the Random Forest rather than Logistic Regression because:
Unlike Logistic Regression (which is linear), the Random Forest is non-linear. It can automatically learn complex rules like: "If the favorite has an ELO advantage, BUT they have bad form AND are missing key players due to injuries, predict a draw/upset.


In [98]:
print(f"Training Advanced Random Forest on {len(full_all)} matches")
rf_base_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_base_model.fit(full_all[full_features], full_all["result"])

Training Advanced Random Forest on 30696 matches


,n_estimators,300
,criterion,'gini'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 5. Symmetric Double-Inference Prediction

we predict every match twice:
1. Forward: Team A vs Team B.
2. Reversed: Team B vs Team A (flipping all ELO/form differences to negative, and swapping streaks).
* It then aligns and averages the probabilities. This ensures that the team listed first gets zero arbitrary home advantage.

In [99]:
print("Running symmetric predictions to remove neutral stadium order bias")

probs_forward = rf_base_model.predict_proba(schedule_model[full_features])
schedule_reversed = schedule_model.copy()
diff_cols = [
    "elo_diff", "fifa_rank_diff", "fifa_points_zscore_diff",
    "form_score_5_diff", "form_score_10_diff", "form_trend_diff",
    "attack_diff", "defense_diff", "rest_days_diff",
    "h2h_wins_diff", "h2h_goals_diff", "h2h_avg_goals_diff",
    "win_streak_diff", "unbeaten_streak_diff",
    "strength_rating_diff", "xg_balance_diff", "missing_player_value_diff"
]
for col in diff_cols:
    if col in schedule_reversed.columns:
        schedule_reversed[col] = -schedule_reversed[col]

schedule_reversed["home_win_streak"], schedule_reversed["away_win_streak"] = schedule_model["away_win_streak"], schedule_model["home_win_streak"]
schedule_reversed["home_unbeaten_streak"], schedule_reversed["away_unbeaten_streak"] = schedule_model["away_unbeaten_streak"], schedule_model["home_unbeaten_streak"]
schedule_reversed["is_home_confed"], schedule_reversed["is_away_confed"] = schedule_model["is_away_confed"], schedule_model["is_home_confed"]

#Swap home country advantage (Canada, Mexico, US playing in their home country)
hosts = ["Mexico", "Canada", "United States"]
schedule_reversed["is_home_country"] = schedule_model.apply(
    lambda r: 1 if r["away_team"] in hosts and r["country"] == r["away_team"] else 0,
    axis=1
)

# run the Random Forest model on the swapped dataframe to get the probabilities of the reversed match.
probs_reversed = rf_base_model.predict_proba(schedule_reversed[full_features])

# Aligning the Results
classes = list(rf_base_model.classes_)
hw_idx = classes.index("home_win")
d_idx = classes.index("draw")
aw_idx = classes.index("away_win")
probs_reversed_aligned = np.zeros_like(probs_reversed)
probs_reversed_aligned[:, hw_idx] = probs_reversed[:, aw_idx]
probs_reversed_aligned[:, aw_idx] = probs_reversed[:, hw_idx]
probs_reversed_aligned[:, d_idx] = probs_reversed[:, d_idx]

# Averaging the Probabilities
rf_probs = (probs_forward + probs_reversed_aligned) / 2.0

Running symmetric predictions to remove neutral stadium order bias


## 6. Apply Override Logic with Safety Block

#### This is a surprise detection filter

In [100]:
# Extract bookmaker implied probabilities
mkt_home = schedule_model["market_home_prob_norm"].values
mkt_draw = schedule_model["market_draw_prob_norm"].values
mkt_away = schedule_model["market_away_prob_norm"].values

# Calculate Edge: (Model Probability - Bookmaker Probability)
edge_home = rf_probs[:, hw_idx] - mkt_home
edge_draw = rf_probs[:, d_idx] - mkt_draw
edge_away = rf_probs[:, aw_idx] - mkt_away

final_preds = []
final_reasons = []
upset_potentials = []

for i in range(len(schedule_model)):
    # Find market favorite
    mkt_probs = {"home_win": mkt_home[i], "draw": mkt_draw[i], "away_win": mkt_away[i]}
    favorite = max(mkt_probs, key=mkt_probs.get)
    
    predicted_result = favorite
    reason = "Market Favorite"
    upset_potential = "Low"
    
    #Disable overrides if the favorite is overwhelming (>= 82% market prob(this is hardcoded above))
    if mkt_probs[favorite] >= MAX_FAVORITE_THRESHOLD:
        reason = "Favorite by far "
        upset_potential = "Low"
    else:
        edges = {"home_win": edge_home[i], "draw": edge_draw[i], "away_win": edge_away[i]}
        fav_edge = edges[favorite]
        if fav_edge < -0.04:
            non_favs = {k: v for k, v in edges.items() if k != favorite}
            best_value_class = max(non_favs, key=non_favs.get)
            best_value = non_favs[best_value_class]
            
            # Check surprise
            if best_value_class == "draw":
                upset_potential = "Medium" if best_value < DRAW_EDGE_THRESHOLD else "High"
                if best_value >= DRAW_EDGE_THRESHOLD:
                    predicted_result = "draw"
                    reason = f"Value Draw (Draw Edge +{best_value:.2f})"
            else:
                # Underdog win
                upset_potential = "Medium" if best_value < UPSET_EDGE_THRESHOLD else "High"
                if best_value >= UPSET_EDGE_THRESHOLD:
                    predicted_result = best_value_class
                    reason = f"Upset Alert (Underdog Edge +{best_value:.2f})"
                
    final_preds.append(predicted_result)
    final_reasons.append(reason)
    upset_potentials.append(upset_potential)

## 7. Compile and Save Output

In [101]:
predictions_calibrated = schedule_2026.copy()
predictions_calibrated["predicted_result"] = final_preds
predictions_calibrated["prediction_reason"] = final_reasons
predictions_calibrated["upset_potential"] = upset_potentials

# probabilities
predictions_calibrated["prob_home_win"] = rf_probs[:, hw_idx]
predictions_calibrated["prob_draw"] = rf_probs[:, d_idx]
predictions_calibrated["prob_away_win"] = rf_probs[:, aw_idx]
predictions_calibrated["prediction_confidence"] = rf_probs.max(axis=1)

# confidence 
def calibrated_tier(row):
    if row["upset_potential"] == "High":
        return "low (High Upset Risk)"
    x = row["prediction_confidence"]
    if x >= 0.60: return "high"
    if x >= 0.48: return "medium"
    return "low"

predictions_calibrated["confidence_tier"] = predictions_calibrated.apply(calibrated_tier, axis=1)
predictions_calibrated = predictions_calibrated.sort_values("date").reset_index(drop=True)

### Save to CSV

In [102]:
output_path = "datasets/wc_2026_stacked_lr_predictions.csv"
predictions_calibrated.to_csv(output_path, index=False)
print(f"Predictions successfully saved to: {output_path}")

Predictions successfully saved to: datasets/wc_2026_stacked_lr_predictions.csv


### Display predicted surprises

In [103]:
from IPython.display import HTML

surprises = predictions_calibrated[predictions_calibrated["upset_potential"] == "High"]
print(f"\nDetected {len(surprises)} matches where a surprise is predicted:")
display(HTML(f"""
<div style="height: 300px; overflow-y: scroll; border: 1px solid #ccc; padding: 10px;">
    {surprises[["date", "home_team", "away_team", "predicted_result", 
                "prediction_reason", "prob_home_win", "prob_draw", "prob_away_win"]].to_html(index=False)}
</div>
"""))

display(HTML(f"""
<div style="height: 400px; overflow-y: scroll; border: 1px solid #ccc; padding: 10px;">
    {predictions_calibrated[['date', 'home_team', 'away_team', 'group', 
                             'predicted_result', 'prediction_reason', 
                             'upset_potential', 'confidence_tier']].to_html(index=False)}
</div>
"""))


Detected 37 matches where a surprise is predicted:


date,home_team,away_team,predicted_result,prediction_reason,prob_home_win,prob_draw,prob_away_win
2026-06-13,Qatar,Switzerland,draw,Value Draw (Draw Edge +0.17),0.161326,0.311332,0.527342
2026-06-13,Brazil,Morocco,draw,Value Draw (Draw Edge +0.12),0.386453,0.381915,0.231632
2026-06-13,Haiti,Scotland,draw,Value Draw (Draw Edge +0.16),0.225701,0.383826,0.390473
2026-06-13,Australia,Turkey,draw,Value Draw (Draw Edge +0.12),0.259927,0.379145,0.360928
2026-06-14,Netherlands,Japan,draw,Value Draw (Draw Edge +0.12),0.346529,0.397005,0.256466
2026-06-15,Belgium,Egypt,draw,Value Draw (Draw Edge +0.13),0.424827,0.366023,0.209150
2026-06-16,France,Senegal,draw,Value Draw (Draw Edge +0.12),0.481763,0.336891,0.181346
2026-06-16,Iraq,Norway,draw,Value Draw (Draw Edge +0.20),0.184281,0.329197,0.486522
2026-06-16,Austria,Jordan,draw,Value Draw (Draw Edge +0.18),0.436773,0.361781,0.201446
2026-06-17,Portugal,DR Congo,draw,Value Draw (Draw Edge +0.14),0.546848,0.311673,0.141478


date,home_team,away_team,group,predicted_result,prediction_reason,upset_potential,confidence_tier
2026-06-11,Mexico,South Africa,J,home_win,Market Favorite,Medium,medium
2026-06-11,South Korea,Czech Republic,J,home_win,Market Favorite,Low,low
2026-06-12,Canada,Bosnia and Herzegovina,D,home_win,Market Favorite,Low,medium
2026-06-12,United States,Paraguay,B,home_win,Market Favorite,Medium,low
2026-06-13,Qatar,Switzerland,D,draw,Value Draw (Draw Edge +0.17),High,low (High Upset Risk)
2026-06-13,Brazil,Morocco,E,draw,Value Draw (Draw Edge +0.12),High,low (High Upset Risk)
2026-06-13,Haiti,Scotland,E,draw,Value Draw (Draw Edge +0.16),High,low (High Upset Risk)
2026-06-13,Australia,Turkey,B,draw,Value Draw (Draw Edge +0.12),High,low (High Upset Risk)
2026-06-14,Germany,Curaçao,I,home_win,Favorite by far,Low,high
2026-06-14,Ivory Coast,Ecuador,I,away_win,Market Favorite,Low,low


# Score prediction 

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

goal_features = [
    "elo_diff", "fifa_rank_diff",
    "attack_diff", "defense_diff",
    "xg_attack_diff", "xg_defense_diff", "xg_balance_diff",
    "strength_rating_diff", "attack_rating_diff", "defense_rating_diff",
    "unavailable_diff", "missing_player_value_diff",
    "market_home_prob_norm", "market_away_prob_norm", "odds_prob_diff",
    "neutral", "is_home_country", "rest_days_diff"
]

## 2. Prepare data

In [ ]:
def prep(df, is_schedule=False):
    df = df.copy()
    for col in goal_features:
        if col not in df.columns:
            df[col] = np.nan
    df["neutral"] = df["neutral"].fillna(1.0 if is_schedule else 0.0)
    df["is_home_country"] = df["is_home_country"].fillna(0.0)
    df[goal_features] = df[goal_features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df

sched  = prep(schedule_2026, is_schedule=True)
train  = prep(full_df, is_schedule=False)
train  = train[(train["date"] < "2026-06-11") & (train["result"].notna())].copy()

### 3. Reversed schedule

In [ ]:
rev = sched.copy()
for col in ["elo_diff","fifa_rank_diff","attack_diff","defense_diff",
            "xg_attack_diff","xg_defense_diff","xg_balance_diff",
            "strength_rating_diff","attack_rating_diff","defense_rating_diff",
            "unavailable_diff","missing_player_value_diff","odds_prob_diff",
            "rest_days_diff"]:
    if col in rev.columns:
        rev[col] = -rev[col]

rev["market_home_prob_norm"], rev["market_away_prob_norm"] = sched["market_away_prob_norm"], sched["market_home_prob_norm"]

### 4. Train goal models

In [ ]:
gbr_h = HistGradientBoostingRegressor(max_iter=100, random_state=42)
gbr_a = HistGradientBoostingRegressor(max_iter=100, random_state=42)
gbr_h.fit(train[goal_features], train["home_score"])
gbr_a.fit(train[goal_features], train["away_score"])

lam_h = ((gbr_h.predict(sched[goal_features]) + gbr_a.predict(rev[goal_features])) / 2.0).clip(0.1, 8.0)
lam_a = ((gbr_a.predict(sched[goal_features]) + gbr_h.predict(rev[goal_features])) / 2.0).clip(0.1, 8.0)

### 5. Build score lookup keyed by (home_team, away_team)

In [ ]:
score_lookup = {}
for i in range(len(sched)):
    home = schedule_2026.iloc[i]["home_team"]
    away = schedule_2026.iloc[i]["away_team"]
    pred = schedule_2026.iloc[i]["home_team"] 

    match = predictions_calibrated[
        (predictions_calibrated["home_team"] == home) &
        (predictions_calibrated["away_team"] == away)
    ]
    if len(match) == 0:
        score_lookup[(home, away)] = "1-0"
        continue

    pred = match.iloc[0]["predicted_result"] 
    rng  = np.random.default_rng(seed=42 + i)
    h, a = 0, 0
    found = False

    for _ in range(500):
        sh = rng.poisson(lam_h[i])
        sa = rng.poisson(lam_a[i])
        if pred == "home_win" and sh > sa: h, a = sh, sa; found = True; break
        if pred == "away_win" and sa > sh: h, a = sh, sa; found = True; break
        if pred == "draw"     and sh == sa: h, a = sh, sa; found = True; break

    if not found:
        h = max(0, int(np.round(lam_h[i])))
        a = max(0, int(np.round(lam_a[i])))
        if   pred == "home_win" and h <= a: h = a + 1
        elif pred == "away_win" and a <= h: a = h + 1
        elif pred == "draw"     and h != a: h = a = max(0, int(np.round((lam_h[i] + lam_a[i]) / 2.0)))

    score_lookup[(home, away)] = f"{h}-{a}"

### 6. Add predicted_score directly to existing predictions_calibrated

In [ ]:
predictions_calibrated["predicted_score"] = predictions_calibrated.apply(
    lambda r: score_lookup.get((r["home_team"], r["away_team"]), "1-0"), axis=1
)

Training Goal Models...
Done! predicted_score added.


### Save


In [ ]:
predictions_calibrated.to_csv("datasets/wc_2026_stacked_lr_predictions.csv", index=False)
print("Done!")

### Print the predicted winner and score for each match

In [ ]:
for idx, row in predictions_calibrated.iterrows():
    # Ignore placeholder matches
    if "Winner" in row["home_team"] or "Runner-up" in row["home_team"] or "3rd" in row["home_team"]:
        continue
        
    print(f"{row['home_team']} vs. {row['away_team']} ➔ Predicted: {row['predicted_result']} | Score: {row['predicted_score']}")

Mexico vs. South Africa ➔ Predicted: home_win | Score: 4-3
South Korea vs. Czech Republic ➔ Predicted: home_win | Score: 1-0
Canada vs. Bosnia and Herzegovina ➔ Predicted: home_win | Score: 2-1
United States vs. Paraguay ➔ Predicted: home_win | Score: 1-0
Qatar vs. Switzerland ➔ Predicted: draw | Score: 1-1
Brazil vs. Morocco ➔ Predicted: draw | Score: 1-1
Haiti vs. Scotland ➔ Predicted: draw | Score: 2-2
Australia vs. Turkey ➔ Predicted: draw | Score: 1-1
Germany vs. Curaçao ➔ Predicted: home_win | Score: 5-0
Ivory Coast vs. Ecuador ➔ Predicted: away_win | Score: 1-3
Netherlands vs. Japan ➔ Predicted: draw | Score: 1-1
Sweden vs. Tunisia ➔ Predicted: home_win | Score: 3-1
Saudi Arabia vs. Uruguay ➔ Predicted: away_win | Score: 1-2
Spain vs. Cape Verde ➔ Predicted: home_win | Score: 2-1
Belgium vs. Egypt ➔ Predicted: draw | Score: 1-1
Iran vs. New Zealand ➔ Predicted: home_win | Score: 2-0
France vs. Senegal ➔ Predicted: draw | Score: 1-1
Iraq vs. Norway ➔ Predicted: draw | Score: 0-0
